# Assignment 1 – Image Classification Pipeline



## Problem Statement


Create a reusable Python pipeline that loads an image dataset, resizes images, normalizes pixel values, splits the data into train/validation sets, and stores metadata in a CSV.





## Dataset Description

The Microsoft Cats vs Dogs dataset is used for this assignment. It is a binary image classification dataset containing images of cats and dogs.

The dataset is organized into two folders:

- Cat
- Dog

Each folder contains JPEG images belonging to its respective class. The dataset is commonly used for image classification and computer vision tasks.

In [1]:
#import the required libs

import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
dataset_path = "/content/drive/MyDrive/PetImages"

print("Dataset Path:", dataset_path)

Dataset Path: /content/drive/MyDrive/PetImages


## Create Reusable Image Preprocessing Pipeline

A reusable preprocessing pipeline is created to automate the following tasks:

- Load images from the dataset
- Skip corrupted images
- Resize images to a fixed size (128 × 128)
- Normalize pixel values
- Store image information as metadata

The same pipeline can be reused with any image dataset having a similar folder structure.

In [8]:
def image_preprocessing_pipeline(dataset_path, image_size=(128, 128), images_per_class=100):

    images = []
    labels = []
    metadata = []

    classes = ["Cat", "Dog"]

    for label in classes:

        folder_path = os.path.join(dataset_path, label)

        count = 0

        for image_name in os.listdir(folder_path):

            if count >= images_per_class:
                break

            image_path = os.path.join(folder_path, image_name)

            try:

                image = cv2.imread(image_path)

                if image is None:
                    continue

                # Resize image
                image = cv2.resize(image, image_size)

                # Normalize image
                image = image.astype(np.float32) / 255.0

                images.append(image)
                labels.append(label)

                metadata.append({
                    "Image Name": image_name,
                    "Label": label,
                    "Image Path": image_path
                })

                count += 1

            except:
                continue

    images = np.array(images)
    labels = np.array(labels)
    metadata = pd.DataFrame(metadata)

    return images, labels, metadata

For faster testing we have implemented limited amount of images with a count variable and maxz limit

In [18]:
# Display Dataset Information

def display_dataset_info(images, labels, metadata):

    print("Total Images Loaded :", len(images))
    print("Total Labels :", len(labels))
    print("Metadata Shape :", metadata.shape)
    print("Image Array Shape :", images.shape)

In [19]:
# Split Dataset

def split_dataset(images, labels, test_size=0.2):

    X_train, X_val, y_train, y_val = train_test_split(
        images,
        labels,
        test_size=test_size,
        random_state=42,
        stratify=labels
    )

    return X_train, X_val, y_train, y_val

In [20]:
# Display Train Validation Information

def display_split_info(X_train, X_val):

    print("Training Images :", len(X_train))
    print("Validation Images :", len(X_val))

In [22]:
# Save Metadata

def save_metadata(metadata, filename="metadata.csv"):
    metadata.to_csv(filename, index=False)
    print(filename, "saved successfully.")

In [23]:
# Display Metadata

def display_metadata(metadata, rows=5):
    return metadata.head(rows)

In [24]:
# Testing the Pipeline

TEST_IMAGES_PER_CLASS = 15

images, labels, metadata = image_preprocessing_pipeline(
    dataset_path,
    images_per_class=TEST_IMAGES_PER_CLASS
)

display_dataset_info(images, labels, metadata)


display_metadata(metadata)

X_train, X_val, y_train, y_val = split_dataset(images, labels)

display_split_info(X_train, X_val)

save_metadata(metadata)

Total Images Loaded : 30
Total Labels : 30
Metadata Shape : (30, 3)
Image Array Shape : (30, 128, 128, 3)
Training Images : 24
Validation Images : 6
metadata.csv saved successfully.


## Observations

- The reusable preprocessing pipeline successfully loaded images from the dataset.
- All images were resized to a fixed size of 128 × 128 pixels.
- Pixel values were normalized to the range [0,1].
- The dataset was successfully divided into training and validation sets using an 80:20 ratio.
- Metadata containing image names, labels, and file paths was generated and stored as a CSV file.
- The modular design allows different image sample sizes to be tested by changing a single configuration variable.